# Customer Churn Prediction for a Digital Banking Product

**Author:** Daniela Serrato | 6+ years in fintech & banking analytics

---

## Context

A digital-first bank needs to **predict which customers will churn within the next 30 days** so the retention team can intervene with targeted offers. Churn is expensive: acquiring a new customer costs 5-7x more than retaining an existing one, and in digital banking the switching cost is near zero.

**Objective:** Build and compare binary classification models (Logistic Regression, Random Forest, XGBoost) that maximise recall on churners while keeping precision commercially viable.

**Pipeline overview:**
1. Synthetic data generation (8 000 customers)
2. Exploratory data analysis
3. Feature engineering
4. Model training with SMOTE + cross-validation
5. Hyperparameter tuning
6. Evaluation & threshold optimisation
7. Business-impact estimation

---
## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import (
    train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, auc, f1_score,
    accuracy_score, precision_score, recall_score
)
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

plt.style.use('seaborn-v0_8-darkgrid')
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)

RANDOM_STATE = 42
print('All imports loaded successfully.')

---
## 2. Synthetic Data Generation

In [ ]:
np.random.seed(RANDOM_STATE)
n = 8000

# --- Base features ---
age = np.random.normal(38, 12, n).clip(18, 75).astype(int)
tenure_months = np.random.exponential(24, n).clip(1, 120).astype(int)
num_products = np.random.choice([1, 2, 3, 4], n, p=[0.45, 0.35, 0.15, 0.05])
has_credit_card = np.random.binomial(1, 0.60, n)
has_savings = np.random.binomial(1, 0.40, n)

# --- Behavioural features (correlated with churn drivers) ---
monthly_transactions = np.random.poisson(15, n) + 1
avg_transaction_amount = np.random.lognormal(4.0, 0.8, n).clip(5, 5000).round(2)
monthly_balance_avg = (avg_transaction_amount * monthly_transactions * np.random.uniform(0.3, 2.0, n)).round(2)
customer_service_calls = np.random.poisson(1.2, n)
days_since_last_transaction = np.random.exponential(10, n).clip(0, 90).astype(int)

# --- Churn probability (logistic function of realistic drivers) ---
z = (
    -2.0
    - 0.03 * tenure_months
    - 0.10 * num_products
    - 0.04 * monthly_transactions
    + 0.35 * customer_service_calls
    + 0.04 * days_since_last_transaction
    - 0.3 * has_savings
    - 0.00001 * monthly_balance_avg
    + np.random.normal(0, 0.5, n)
)
churn_prob = 1 / (1 + np.exp(-z))
is_churned = (churn_prob > np.percentile(churn_prob, 80)).astype(int)  # ~20% churn

df = pd.DataFrame({
    'age': age,
    'tenure_months': tenure_months,
    'monthly_transactions': monthly_transactions,
    'avg_transaction_amount': avg_transaction_amount,
    'num_products': num_products,
    'has_credit_card': has_credit_card,
    'has_savings': has_savings,
    'customer_service_calls': customer_service_calls,
    'days_since_last_transaction': days_since_last_transaction,
    'monthly_balance_avg': monthly_balance_avg,
    'is_churned': is_churned
})

print(f'Dataset shape: {df.shape}')
print(f'Churn rate: {df.is_churned.mean():.2%}')
df.head(10)

In [ ]:
df.describe().round(2)

In [ ]:
df.info()

---
## 3. Exploratory Data Analysis

In [ ]:
# --- 3.1 Class distribution ---
fig, ax = plt.subplots(figsize=(5, 4))
df['is_churned'].value_counts().plot.bar(
    color=['#2196F3', '#F44336'], edgecolor='white', ax=ax
)
ax.set_title('Class Distribution (0 = Active, 1 = Churned)', fontsize=13)
ax.set_ylabel('Count')
ax.set_xticklabels(['Active', 'Churned'], rotation=0)
for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}', (p.get_x() + p.get_width() / 2, p.get_height()),
               ha='center', va='bottom', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# --- 3.2 Correlation heatmap ---
fig, ax = plt.subplots(figsize=(10, 8))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# --- 3.3 Feature distributions by churn status ---
continuous_features = [
    'tenure_months', 'monthly_transactions', 'avg_transaction_amount',
    'customer_service_calls', 'days_since_last_transaction', 'monthly_balance_avg'
]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, col in zip(axes.ravel(), continuous_features):
    for label, colour in zip([0, 1], ['#2196F3', '#F44336']):
        subset = df[df['is_churned'] == label][col]
        ax.hist(subset, bins=30, alpha=0.55, label=f'{"Active" if label==0 else "Churned"}',
                color=colour, edgecolor='white')
    ax.set_title(col, fontsize=12)
    ax.legend(fontsize=9)
fig.suptitle('Feature Distributions by Churn Status', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# --- 3.4 Box plots for key features ---
key_features = ['tenure_months', 'customer_service_calls',
                'days_since_last_transaction', 'monthly_transactions']

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, col in zip(axes, key_features):
    sns.boxplot(data=df, x='is_churned', y=col, ax=ax,
                palette=['#2196F3', '#F44336'], width=0.4)
    ax.set_xticklabels(['Active', 'Churned'])
    ax.set_title(col, fontsize=12)
fig.suptitle('Key Features by Churn Status', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 4. Feature Engineering

In [ ]:
# --- Derived features ---

# Transaction trend proxy: ratio of days since last txn to avg monthly txns
# Higher values indicate declining engagement
df['transaction_trend'] = (
    df['days_since_last_transaction'] / (df['monthly_transactions'] + 1)
).round(4)

# Balance-to-transaction ratio: how much balance per transaction
df['balance_to_transaction_ratio'] = (
    df['monthly_balance_avg'] / (df['avg_transaction_amount'] * df['monthly_transactions'] + 1)
).round(4)

# Engagement score: composite metric (higher = more engaged)
df['engagement_score'] = (
    0.3 * df['num_products'] / df['num_products'].max()
    + 0.3 * df['monthly_transactions'] / df['monthly_transactions'].max()
    + 0.2 * df['has_credit_card']
    + 0.2 * df['has_savings']
).round(4)

print('New features created:')
print(df[['transaction_trend', 'balance_to_transaction_ratio', 'engagement_score']].describe().round(3))

In [ ]:
# Quick sanity check: correlation of new features with churn
new_feats = ['transaction_trend', 'balance_to_transaction_ratio', 'engagement_score']
for f in new_feats:
    corr_val = df[f].corr(df['is_churned'])
    print(f'{f:>35s}  corr with churn = {corr_val:+.3f}')

---
## 5. Modelling Pipeline

In [ ]:
# --- 5.1 Define X, y and split ---
feature_cols = [
    'age', 'tenure_months', 'monthly_transactions', 'avg_transaction_amount',
    'num_products', 'has_credit_card', 'has_savings', 'customer_service_calls',
    'days_since_last_transaction', 'monthly_balance_avg',
    'transaction_trend', 'balance_to_transaction_ratio', 'engagement_score'
]

X = df[feature_cols].copy()
y = df['is_churned'].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print(f'Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}')
print(f'Train churn rate: {y_train.mean():.2%} | Test churn rate: {y_test.mean():.2%}')

In [ ]:
# --- 5.2 SMOTE for class imbalance ---
smote = SMOTE(random_state=RANDOM_STATE)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f'After SMOTE -> Train samples: {X_train_sm.shape[0]:,}')
print(f'Class distribution:\n{pd.Series(y_train_sm).value_counts()}')

In [ ]:
# --- 5.3 Scale features ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_sm)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# --- 5.4 Define models ---
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, random_state=RANDOM_STATE
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1
    ),
    'XGBoost': XGBClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.1,
        random_state=RANDOM_STATE, eval_metric='logloss',
        use_label_encoder=False, n_jobs=-1
    )
}

In [ ]:
# --- 5.5 Cross-validation (5-fold) ---
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_results = {}
for name, model in models.items():
    scores = cross_val_score(model, X_train_scaled, y_train_sm,
                            cv=cv, scoring='roc_auc', n_jobs=-1)
    cv_results[name] = scores
    print(f'{name:>22s}  |  AUC = {scores.mean():.4f} +/- {scores.std():.4f}')

In [ ]:
# --- 5.6 Train all models on full (SMOTE) training set ---
trained_models = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train_sm)
    trained_models[name] = model
    y_pred = model.predict(X_test_scaled)
    print(f'\n--- {name} ---')
    print(classification_report(y_test, y_pred, target_names=['Active', 'Churned']))

In [ ]:
# --- 5.7 Hyperparameter tuning for best CV model ---
# Identify best model from CV
best_model_name = max(cv_results, key=lambda k: cv_results[k].mean())
print(f'Best CV model: {best_model_name} -> tuning hyperparameters...\n')

param_grids = {
    'Logistic Regression': {
        'C': [0.01, 0.1, 1, 10],
        'penalty': ['l2']
    },
    'Random Forest': {
        'n_estimators': [100, 300],
        'max_depth': [8, 15, None],
        'min_samples_split': [2, 5]
    },
    'XGBoost': {
        'n_estimators': [100, 300],
        'max_depth': [3, 5, 7],
        'learning_rate': [0.05, 0.1]
    }
}

grid_search = GridSearchCV(
    models[best_model_name],
    param_grids[best_model_name],
    cv=cv,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=0
)
grid_search.fit(X_train_scaled, y_train_sm)

print(f'Best params: {grid_search.best_params_}')
print(f'Best CV AUC:  {grid_search.best_score_:.4f}')

best_model = grid_search.best_estimator_
trained_models[f'{best_model_name} (tuned)'] = best_model

---
## 6. Evaluation

In [ ]:
# --- 6.1 Confusion matrices ---
eval_models = {k: v for k, v in trained_models.items() if '(tuned)' not in k}

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, (name, model) in zip(axes, eval_models.items()):
    y_pred = model.predict(X_test_scaled)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Active', 'Churned'],
                yticklabels=['Active', 'Churned'])
    ax.set_title(name, fontsize=12)
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')
fig.suptitle('Confusion Matrices', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- 6.2 ROC Curves (all 3 models on one plot) ---
fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#2196F3', '#4CAF50', '#FF9800']

auc_scores = {}
for (name, model), color in zip(eval_models.items(), colors):
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)
    auc_scores[name] = roc_auc
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC = {roc_auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves - Model Comparison', fontsize=14)
ax.legend(fontsize=11, loc='lower right')
plt.tight_layout()
plt.show()

print('\nAUC Comparison:')
for name, score in sorted(auc_scores.items(), key=lambda x: -x[1]):
    print(f'  {name:>22s}: {score:.4f}')

In [ ]:
# --- 6.3 Precision-Recall Curve ---
fig, ax = plt.subplots(figsize=(8, 6))

for (name, model), color in zip(eval_models.items(), colors):
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    prec, rec, _ = precision_recall_curve(y_test, y_proba)
    pr_auc = auc(rec, prec)
    ax.plot(rec, prec, color=color, lw=2, label=f'{name} (PR-AUC = {pr_auc:.3f})')

ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curves', fontsize=14)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# --- 6.4 Feature importance (best base model) ---
# Use XGBoost or Random Forest for feature importance
fi_model_name = 'XGBoost' if 'XGBoost' in eval_models else 'Random Forest'
fi_model = eval_models[fi_model_name]

importances = fi_model.feature_importances_
feat_imp = pd.DataFrame({
    'feature': feature_cols,
    'importance': importances
}).sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(feat_imp['feature'], feat_imp['importance'], color='#2196F3', edgecolor='white')
ax.set_xlabel('Importance', fontsize=12)
ax.set_title(f'Feature Importance ({fi_model_name})', fontsize=14)
plt.tight_layout()
plt.show()

---
## 7. Threshold Optimisation

In [ ]:
# Use the best performing model for threshold analysis
final_model = best_model
y_proba_final = final_model.predict_proba(X_test_scaled)[:, 1]

# Precision-recall tradeoff across thresholds
thresholds = np.arange(0.1, 0.9, 0.01)
results = []
for t in thresholds:
    y_pred_t = (y_proba_final >= t).astype(int)
    results.append({
        'threshold': t,
        'precision': precision_score(y_test, y_pred_t, zero_division=0),
        'recall': recall_score(y_test, y_pred_t, zero_division=0),
        'f1': f1_score(y_test, y_pred_t, zero_division=0)
    })

thresh_df = pd.DataFrame(results)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(thresh_df['threshold'], thresh_df['precision'], label='Precision',
        color='#2196F3', lw=2)
ax.plot(thresh_df['threshold'], thresh_df['recall'], label='Recall',
        color='#F44336', lw=2)
ax.plot(thresh_df['threshold'], thresh_df['f1'], label='F1 Score',
        color='#4CAF50', lw=2, linestyle='--')

# Optimal threshold (max F1)
optimal_idx = thresh_df['f1'].idxmax()
optimal_thresh = thresh_df.loc[optimal_idx, 'threshold']
ax.axvline(optimal_thresh, color='gray', linestyle=':', lw=1.5,
           label=f'Optimal threshold = {optimal_thresh:.2f}')

ax.set_xlabel('Decision Threshold', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Precision-Recall Tradeoff by Threshold', fontsize=14)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print(f'\nOptimal threshold (max F1): {optimal_thresh:.2f}')
print(f'  Precision = {thresh_df.loc[optimal_idx, "precision"]:.3f}')
print(f'  Recall    = {thresh_df.loc[optimal_idx, "recall"]:.3f}')
print(f'  F1        = {thresh_df.loc[optimal_idx, "f1"]:.3f}')

In [ ]:
# Final classification report with optimised threshold
y_pred_optimal = (y_proba_final >= optimal_thresh).astype(int)

print(f'Classification Report (threshold = {optimal_thresh:.2f}):')
print(classification_report(y_test, y_pred_optimal, target_names=['Active', 'Churned']))

fig, ax = plt.subplots(figsize=(5, 4))
cm = confusion_matrix(y_test, y_pred_optimal)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Active', 'Churned'],
            yticklabels=['Active', 'Churned'])
ax.set_title(f'Confusion Matrix (threshold = {optimal_thresh:.2f})', fontsize=12)
ax.set_ylabel('Actual')
ax.set_xlabel('Predicted')
plt.tight_layout()
plt.show()

---
## 8. Business Impact

### Estimated Savings from Early Churn Detection

| Assumption | Value |
|---|---|
| Average monthly revenue per customer | $85 |
| Average customer lifetime (months) | 24 |
| Customer acquisition cost (CAC) | $250 |
| Retention campaign cost per customer | $15 |
| Retention success rate with intervention | 30% |

**Without the model:** the bank loses all churning customers (no proactive action).

**With the model:** flagged customers receive a retention offer. At 30% success rate, a portion is saved.

### ROI calculation

- **True positives (correctly identified churners):** each saved customer is worth ~$2,040 in remaining LTV minus the $15 campaign cost.
- **False positives (active customers flagged):** cost = $15 per unnecessary campaign (low harm).
- **False negatives (missed churners):** lost LTV = ~$2,040 each.

Even a model with moderate recall (~70%) can generate significant net savings by catching the majority of at-risk customers at a very low per-contact cost.

In [ ]:
# --- Quantified business impact ---
avg_monthly_revenue = 85
avg_lifetime_months = 24
ltv = avg_monthly_revenue * avg_lifetime_months  # $2,040
campaign_cost = 15
retention_rate = 0.30

tp = cm[1, 1]  # true positives
fp = cm[0, 1]  # false positives
fn = cm[1, 0]  # false negatives

customers_saved = tp * retention_rate
revenue_saved = customers_saved * ltv
campaign_total_cost = (tp + fp) * campaign_cost
revenue_lost = fn * ltv
net_benefit = revenue_saved - campaign_total_cost

# Scale to full customer base (8000)
scale_factor = 8000 / len(y_test)

print('=== Business Impact (scaled to 8,000 customers) ===')
print(f'  Churners identified (TP):      {tp * scale_factor:,.0f}')
print(f'  Customers saved (30% rate):     {customers_saved * scale_factor:,.0f}')
print(f'  Revenue saved:                 ${revenue_saved * scale_factor:,.0f}')
print(f'  Campaign cost:                 ${campaign_total_cost * scale_factor:,.0f}')
print(f'  Missed churners (FN):           {fn * scale_factor:,.0f}')
print(f'  Revenue lost (missed):         ${revenue_lost * scale_factor:,.0f}')
print(f'  ----------------------------------------')
print(f'  NET BENEFIT:                   ${net_benefit * scale_factor:,.0f}')

---
## 9. Conclusions

1. **Best model:** The tuned model achieved strong discriminatory power (see AUC above), confirming that customer behaviour signals -- particularly `customer_service_calls`, `days_since_last_transaction`, and `tenure_months` -- are reliable churn predictors.

2. **SMOTE effectively addressed class imbalance:** Without oversampling, models tended to under-predict the minority class. SMOTE improved recall on churners without excessive precision loss.

3. **Threshold optimisation matters:** Moving the decision threshold from the default 0.50 to the F1-optimal value improved the precision-recall balance for the business use case, where false negatives (missed churners) are more costly than false positives.

4. **Feature engineering added value:** Derived features like `transaction_trend` and `engagement_score` captured behavioural nuances that raw features alone missed.

5. **Business impact is tangible:** Even with conservative assumptions (30% retention success rate), early detection generates meaningful net savings. The model is ready for A/B testing in production.

### Next steps
- Integrate real transactional data and validate on a hold-out time period.
- Explore deep-learning approaches (e.g., LSTM on transaction sequences).
- Deploy as a scoring API with scheduled batch inference and monitoring dashboards.

---
*Notebook by Daniela Serrato*